In [21]:
import pandas as pd                 # Data table (jaise Excel) handle karne ke liye
import numpy as np                  # Math calculations ke liye

from sklearn.model_selection import train_test_split       # Data ko train/test mein baantne ke liye
from sklearn.ensemble import RandomForestRegressor          # Price predict karne wala model (best fit isliye chuna)
from sklearn.metrics import mean_absolute_error, r2_score   # Model ki accuracy check karne ke liye

In [22]:
df = pd.read_csv("bengaluru_house_prices.csv")   # CSV file load ki
df.shape   # Kitni rows aur columns hain, check kiya

(13320, 9)

In [23]:
df.info()   # Har column ka data type aur non-null count dekha

<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  str    
 1   availability  13320 non-null  str    
 2   location      13319 non-null  str    
 3   size          13304 non-null  str    
 4   society       7818 non-null   str    
 5   total_sqft    13320 non-null  str    
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), str(6)
memory usage: 936.7 KB


In [24]:
df.isnull().sum()   # Har column mein kitni missing values hain, wo count kiya

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [25]:
# Ye 4 columns price predict karne mein kaam ke nahi hain, isliye hata diye
df = df.drop(['area_type', 'availability', 'society', 'balcony'], axis=1)

df = df.dropna()   # Jis row mein bhi missing value hai, wo row hata di
df.shape            # Cleaning ke baad naya shape check kiya

(13246, 5)

In [26]:
# "2 BHK" jaisa text tha, space se tod kar sirf "2" nikala aur number bana diya
df['bhk'] = df['size'].str.split(' ').str[0].astype(int)

df = df.drop('size', axis=1)   # Purana text wala column ab zaroorat nahi, hata diya
df.head()   # Check kiya ki 'bhk' column sahi bana hai

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056,2.0,39.07,2
1,Chikka Tirupathi,2600,5.0,120.00,4
2,Uttarahalli,1440,2.0,62.00,3
3,Lingadheeranahalli,1521,3.0,95.00,3
4,Kothanur,1200,2.0,51.00,2


In [27]:
def convert_sqft(x):
    tokens = str(x).split('-')          # "2100-2850" jaisi value ko '-' se tod diya
    if len(tokens) == 2:                # Agar range hai (2 tukde bane)
        return (float(tokens[0]) + float(tokens[1])) / 2   # Dono ka average nikal diya
    try:
        return float(x)                 # Normal number ho to seedha convert kar diya
    except:
        return None                     # Convert nahi hua to None bana diya

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)   # Har row pe ye function chalaya
df = df.dropna()    # Jo convert nahi ho payi, wo rows hata di
df.shape

(13200, 5)

In [28]:
df['location'] = df['location'].apply(lambda x: str(x).strip())   # Extra spaces hata diye

# Jin locations ke 10 se kam ghar hain, unhe ek saath "other" naam de diya
counts = df['location'].value_counts()
rare = counts[counts <= 10].index
df['location'] = df['location'].apply(lambda x: 'other' if x in rare else x)

df['location'].nunique()   # Ab kitni alag locations bachi, check kiya

241

In [29]:
# Price per sqft banaya - sirf outliers dhundhne ke liye, model training mein use nahi hoga
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft']

# Ek room ke liye minimum 300 sqft hona chahiye, warna galat entry maani
df = df[df['total_sqft'] / df['bhk'] >= 300]
df.shape

(12456, 6)

In [30]:
def remove_pps_outliers(data):
    cleaned = pd.DataFrame()   # Khali table banayi jisme saaf data jama karenge
    for location, sub_df in data.groupby('location'):   # Har location ka data alag nikala
        m = sub_df['price_per_sqft'].mean()   # Us location ka average price/sqft
        s = sub_df['price_per_sqft'].std()    # Price kitna faila hua hai
        # Sirf wahi ghar rakhe jo average ke ek std-dev ke andar hain
        keep = sub_df[(sub_df['price_per_sqft'] > (m - s)) & (sub_df['price_per_sqft'] <= (m + s))]
        cleaned = pd.concat([cleaned, keep], ignore_index=True)
    return cleaned

df = remove_pps_outliers(df)   # Function chalaya
df.shape

(10293, 6)

In [31]:
# Agar bathroom BHK+2 se zyada hain, to ye data entry error hai
df = df[df['bath'] < df['bhk'] + 2]

# price_per_sqft ab hata diya, kyunki wo price se hi bana tha (training mein rakhna galat hoga)
df = df.drop('price_per_sqft', axis=1)
df.shape

(10198, 5)

In [32]:
# Location (text) ko one-hot encoding se number columns mein badal diya
df = pd.concat([df.drop('location', axis=1),
                 pd.get_dummies(df['location'], drop_first=True).astype(int)], axis=1)

df.shape

(10198, 244)

In [33]:
X = df.drop('price', axis=1)   # Price ke alawa sab features (input) hain
y = df['price']                 # Price = target (jo predict karna hai)

# Data ko 80% training aur 20% testing mein baant diya
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (8158, 243)
Testing: (2040, 243)


In [34]:
model = RandomForestRegressor(n_estimators=100, random_state=42)   # Model banaya (100 trees)
model.fit(X_train, y_train)   # Training data se model ne seekha

y_train_pred = model.predict(X_train)   # Training data pe prediction
y_test_pred = model.predict(X_test)     # Testing data pe prediction (jo model ne kabhi nahi dekha)

In [35]:
train_r2 = r2_score(y_train, y_train_pred)          # Training accuracy
train_mae = mean_absolute_error(y_train, y_train_pred)   # Training average error

test_r2 = r2_score(y_test, y_test_pred)             # Testing accuracy
test_mae = mean_absolute_error(y_test, y_test_pred)      # Testing average error

print("Training R2:", round(train_r2*100, 2), "%   MAE:", round(train_mae, 2), "Lakhs")
print("Testing R2:", round(test_r2*100, 2), "%   MAE:", round(test_mae, 2), "Lakhs")

Training R2: 95.01 %   MAE: 8.56 Lakhs
Testing R2: 75.79 %   MAE: 17.38 Lakhs
